<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c38-header.png" alt="Nextia Learning · Search and Ranking: Find the Best Match" width="100%">

# Solution: Fix spelling

**[Fix spelling](https://learning.nextia-ai.com/courses/search/m04/fix-spelling/)** · Search and Ranking: Find the Best Match · Module 4, lesson 2 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** build a spelling corrector from edit distance and a dictionary, and measure which queries it helps and which it hurts.

| | |
|---|---|
| **Time** | About 35 minutes |
| **Needs** | An internet connection for the setup cells (about 10 MB). No account, no key and no model download. |
| **You should know** | Normalising and stemming, from [Normalise text](https://learning.nextia-ai.com/courses/search/m04/normalise-text/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, pandas 3.1.1 and matplotlib 3.11.2 (bm25s 0.3.16, PyStemmer 3.1.0, sentence-transformers 6.2.0, PyTorch 2.15.0, scikit-learn 1.9.1 and LightGBM 4.7.0 where the lesson needs them), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/search/m04/fix-spelling/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C38/M04-L02-fix-spelling/fix-spelling-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, pandas 3.1.1, bm25s 0.3.16, PyStemmer 3.1.0, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 pandas==3.1.1 matplotlib==3.11.2 bm25s==0.3.16 PyStemmer==3.1.0
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'pandas', 'bm25s', 'PyStemmer']:
    print(name, version(name))

## Setup: get the Larkfield data

Run the next cell. It downloads the files that this lesson needs (about 10 MB) into a folder `larkfield-search`, with the course's helper `compare.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-search` (in Colab: the **Files** panel on the left), then run the cell again.

Where the data comes from: Larkfield is an invented shop. Its products, queries and relevance judgments are derived from WANDS, the Wayfair ANnotation DataSet (MIT licence). The query log with its clicks, and the prices, stock and sale flags, are **synthetic**. The [dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C38/data/dataset-card.md) says what is real and what is made.

In [ ]:
# Setup: download the files into larkfield-search/ and check them.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C38/"
FILES = {
    "data/catalogue.csv.gz": "569c1c4c41403d59492e4cb22be61a7e634b052f67b4d32d12a9f14e77c08c52",
    "data/queries.csv": "51e0286150a6162502faf6502e90b89e4984949ddcb64f1e83f7e1d3251d8634",
    "data/judgments.csv.gz": "6a0b896c64a0e1000fe9401bfc359ae7995732af458120963408d71e50333ec6",
    "data/log/searches.csv.gz": "73da1fb11cebb3891cabca5ab320be2147d1855b9f8e53890007b19fb78862ea",
    "compare.py": "f09c610e8e9026e0c355f693cb56cd3e178e7217c0e001a744280fb82ff867d2",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-search").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-search and run again.")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in larkfield-search.")

## Setup: helpers

The next cell loads NumPy as `np`, pandas as `pd`, matplotlib, and the course helper as `cp` (NDCG, recall, MRR, intervals and the comparison table). It defines a few small helpers:

- `catalogue()`, `queries()` and `judgments()` read the data: one row per product, one row per query, and `{query_id: {product_id: grade}}` with grade 2 exact, 1 partial, 0 irrelevant.
- `show(ids)` prints products with their title and class.

It also defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the data, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import compare as cp
warnings.filterwarnings("ignore")

def catalogue():
    """One row per product."""
    df = pd.read_csv("data/catalogue.csv.gz", keep_default_na=False, dtype={"title_fr": str})
    for col in ("on_sale", "in_stock"):
        df[col] = df[col].astype(str).eq("True")
    return df

def queries():
    """One row per query with judgments (train, validation and test)."""
    return pd.read_csv("data/queries.csv", keep_default_na=False)

def judgments():
    """{query_id: {product_id: grade}}: 2 exact, 1 partial, 0 irrelevant."""
    j = pd.read_csv("data/judgments.csv.gz")
    out = {}
    for q, p, g in zip(j["query_id"], j["product_id"], j["grade"]):
        out.setdefault(q, {})[p] = int(g)
    return out

def show(ids, judged=None, n=10):
    """Print products: rank, ID, title, class (and the grade when judged is given)."""
    global _CAT
    if "_CAT" not in globals():
        _CAT = catalogue().set_index("product_id")
    cat = _CAT
    names = {2: "exact", 1: "partial", 0: "irrelevant"}
    for i, p in enumerate(list(ids)[:n], 1):
        g = "" if judged is None else f"  [{names.get(judged.get(p), 'not judged')}]"
        print(f"{i:2}. {p}  {cat.loc[p, 'title'][:55]:<55}  {cat.loc[p, 'product_class'][:22]}{g}")

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The words that Larkfield knows

A spelling corrector needs a **dictionary**: the words that the shop knows, with a count of how often each appears. The project builds it from the product titles, classes, categories, attributes and French titles, plus the words of the logged searches that got a click. (The log is synthetic: our code made it.) Run the next cell.

In [ ]:
import re, string, unicodedata
from collections import Counter
import Stemmer
EN = Stemmer.Stemmer("english")

def plain(text):
    return [t for t in re.split(r"[^\w]+|_", str(text).lower()) if t]

def fold(text):
    """Lower case, no accents, inches written one way (from the lesson Normalise text)."""
    t = unicodedata.normalize("NFKD", str(text).lower())
    t = "".join(c for c in t if not unicodedata.combining(c))
    return re.sub(r"(\d)\s*(?:''|\"|-?inch(?:es)?\b|in\b)", r"\1 inch ", t)

cat = catalogue()
words = Counter()
for col in ("title", "product_class", "category", "attributes", "title_fr"):
    for text in cat[col]:
        words.update(plain(fold(text)))
log = pd.read_csv("data/log/searches.csv.gz", keep_default_na=False)   # the synthetic query log
for text in log.loc[log["clicks"] > 0, "query"]:                        # only searches that got a click
    words.update(plain(fold(text)))
print(len(words), "words in the dictionary;", "outdoor:", words["outdoor"], " outdor:", words["outdor"])

> **Check.** About 29362 words. (The project's dictionary has 29,312: it writes units in a few more ways, so its tokens differ a little.) "outdoor" appears thousands of times; "outdor" never. A click says the search found something, so its words are probably right; a search with no click may contain the typo that you want to correct.

## 2. Your turn: every string one edit away

One **edit** is a delete, a swap of two neighbour letters, a change of one letter, or an added letter. The **edit distance** between two words is the smallest number of edits from one to the other. Complete `edits1`: the deletes and swaps are written; write the changes and the adds in the same style.

In [ ]:
LETTERS = string.ascii_lowercase

def edits1(w):
    """Every string one edit away from w: delete, swap two neighbours, change or add one letter."""
    splits = [(w[:i], w[i:]) for i in range(len(w) + 1)]
    deletes = {a + b[1:] for a, b in splits if b}
    swaps = {a + b[1] + b[0] + b[2:] for a, b in splits if len(b) > 1}
    changes = {a + c + b[1:] for a, b in splits if b for c in LETTERS}
    adds = {a + c + b for a, b in splits for c in LETTERS}
    return deletes | swaps | changes | adds

n_outdor = len(edits1("outdor"))
print(n_outdor, "strings one edit away from outdor;", sorted(w for w in edits1("outdor") if w in words))

In [ ]:
expect('n_outdor', n_outdor, 338)

> **Check.** 338 strings, and only one of them is in the dictionary: "outdoor".

## 3. The corrector

The next cell is the project's rule. A known word, a short word or a number stays. Otherwise the corrector takes the most frequent dictionary word one edit away, or two edits away for long words.

> **Predict.** What does it do with "filaret" (a sofa collection) and "togo" (a famous chair)?

In [ ]:
def correct_word(w):
    """The project's rule: keep known words, short words and numbers; else the most frequent word 1 edit away,
    or 2 edits away for words of 7 letters or more."""
    if not w.isalpha() or len(w) < 4 or words.get(w, 0) > 0:
        return w
    cands = [c for c in edits1(w) if c in words]
    if not cands and len(w) >= 7:
        cands = [c2 for c1 in edits1(w) for c2 in edits1(c1) if c2 in words]
    return max(cands, key=lambda c: (words[c], c)) if cands else w

def correct(query):
    return " ".join(correct_word(w) for w in plain(fold(query)))

for q in ["outdor cushion", "bar stol with backrest", "filaret outdoor sofa", "togo chair"]:
    print(f"{q:28} -> {correct(q)}")

> **Check.** "outdor" and "stol" are fixed. "filaret" becomes "flared" and "togo" becomes "toro": names that the dictionary does not know become other words.

## 4. A search to measure with

The next cell builds one BM25 index of the title and description, normalised and stemmed as in the lesson *Normalise text*. It takes less than a minute.

In [ ]:
import bm25s
def norm(text):
    return EN.stemWords(plain(fold(text)))
docs = (cat["title"] + " " + cat["description"]).tolist()
index = bm25s.BM25(k1=1.2, b=0.75, method="lucene")
index.index([norm(d) for d in docs], show_progress=False)
ids = cat["product_id"].to_numpy()

def search(query, k=10):
    terms = [w for w in norm(query) if w in index.vocab_dict]
    if not terms:
        return []
    s = index.get_scores(terms)
    top = np.argsort(-s)[:k]
    return [ids[i] for i in top if s[i] > 0]
print("indexed", len(docs), "products")

## 5. Your turn: measure the corrector

Search every English validation query (the real WANDS queries and our planted misspellings) as typed and after correction. Compute the mean NDCG@10 of both.

In [ ]:
q = queries()
val = q[(q["split"] == "validation") & (q["source"].isin(["wands", "typo"]))]
j = judgments()
fixed = {qid: correct(text) for qid, text in zip(val["query_id"], val["query"])}
changed = [qid for qid, text in zip(val["query_id"], val["query"]) if fixed[qid] != " ".join(plain(fold(text)))]
run_before = {qid: search(text) for qid, text in zip(val["query_id"], val["query"])}
run_after = {qid: search(fixed[qid]) for qid in val["query_id"]}
before = cp.per_query(run_before, j)["ndcg@10"]
after = cp.per_query(run_after, j)["ndcg@10"]
print(len(changed), "queries changed; mean NDCG@10", round(before.mean(), 3), "->", round(after.mean(), 3))

In [ ]:
expect('mean NDCG@10 after correction', None if after is ... else after.mean(), 0.673, tolerance=0.002)

> **Check.** 24 of the 80 queries change, and the mean goes from about 0.638 to about 0.673.

## 6. Who gets better, who gets worse?

> **Predict.** The corrector was built for typing errors. Does it help the real WANDS queries as much as the planted misspellings?

In [ ]:
if after is ...:
    print("Not yet: complete the task above first.")
else:
    d = (after - before).loc[changed]
    table = pd.DataFrame({"query": val.set_index("query_id").loc[changed, "query"], "corrected": [fixed[i] for i in changed],
                          "source": val.set_index("query_id").loc[changed, "source"], "change": d.round(3)})
    print(table.groupby("source")["change"].agg(better=lambda s: int((s > 0).sum()), worse=lambda s: int((s < 0).sum()),
                                                same=lambda s: int((s == 0).sum())))
    n_worse = int((d < 0).sum())
    print(table.sort_values("change").head(6).to_string())

> **Check.** 9 queries better and 6 worse. On the planted misspellings, 8 better and 5 worse; on the real WANDS queries, 1 better and 1 worse. The validation split is small: the lesson's 364 validation and test queries show the pattern more clearly (on the real queries, 3 better and 7 worse).
>
> Not every "worse" is a wrong correction. "ayesha cuurry kitchen" becomes the right text, "ayesha curry kitchen", and still scores lower than with the unknown word ignored. Read the worse queries one by one before you blame the corrector.

In the project: `search compare bm25-norm bm25-norm+spell --by source`.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Synonyms and query rewriting](https://learning.nextia-ai.com/courses/search/m04/synonyms-and-query-rewriting/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/search/m04/fix-spelling/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The products, queries and judgments come from WANDS (Wayfair, MIT licence). The query log, prices, stock and sale flags are synthetic: our code made them. No real customer is in this data.